In [ ]:
# Install jika belum ada
!pip install matplotlib opencv-python-headless lxml ultralytics gradio -q

In [ ]:
import os, glob, random
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from ultralytics import YOLO
import shutil
from PIL import Image as PILImage

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
VOC_ROOT = '/content/drive/MyDrive/imageeeeee tes/VOCtrainval_11-May-2012/VOCdevkit/VOC2012'

IMG_DIR = os.path.join(VOC_ROOT, 'JPEGImages')
ANN_DIR = os.path.join(VOC_ROOT, 'Annotations')


print("VOC_ROOT ada:", os.path.exists(VOC_ROOT))
print("JPEGImages ada:", os.path.exists(IMG_DIR))
print("Annotations ada:", os.path.exists(ANN_DIR))


imgs = glob.glob(os.path.join(IMG_DIR, '*.jpg'))
anns = glob.glob(os.path.join(ANN_DIR, '*.xml'))
print(f"\nGambar : {len(imgs)}")
print(f"Anotasi: {len(anns)}")

In [ ]:
def parse_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    objects = []
    for obj in root.findall('object'):
        label = obj.find('name').text
        bb    = obj.find('bndbox')
        xmin  = int(bb.find('xmin').text)
        ymin  = int(bb.find('ymin').text)
        xmax  = int(bb.find('xmax').text)
        ymax  = int(bb.find('ymax').text)
        objects.append((label, xmin, ymin, xmax, ymax))
    return objects

In [ ]:
from collections import Counter
import numpy as np

anns = anns[:1000]

all_labels = []
all_widths = []
all_heights = []
img_object_counts = []

for xml_pe in anns:
    objs = parse_xml(xml_pe)
    img_object_counts.append(len(objs))

    for label, xmin, ymin, xmax, ymax in objs:
        all_labels.append(label)
        all_widths.append(xmax - xmin)
        all_heights.append(ymax - ymin)

label_counts = Counter(all_labels)

print("Total objek   :", len(all_labels))
print("Total class   :", len(label_counts))
print("Class         :", sorted(label_counts))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Bar chart jumlah objek per class
labels_sorted = sorted(label_counts, key=label_counts.get, reverse=True)
counts_sorted = [label_counts[l] for l in labels_sorted]
colors = plt.cm.tab20.colors[:len(labels_sorted)]

axes[0].barh(labels_sorted, counts_sorted, color=colors)
axes[0].set_xlabel('Jumlah Objek')
axes[0].set_title('Distribusi Class (jumlah objek)')
axes[0].invert_yaxis()
for i, v in enumerate(counts_sorted):
    axes[0].text(v + 50, i, str(v), va='center', fontsize=9)

# Pie chart proporsi
axes[1].pie(counts_sorted, labels=labels_sorted, colors=colors,
            autopct='%1.1f%%', startangle=140, textprops={'fontsize': 8})
axes[1].set_title('Proporsi Class')

plt.suptitle('EDA — Distribusi Class Pascal VOC', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram lebar box
axes[0].hist(all_widths, bins=50, color='#4363d8', edgecolor='white')
axes[0].set_title('Distribusi Lebar Bounding Box')
axes[0].set_xlabel('Lebar (pixel)')
axes[0].set_ylabel('Frekuensi')
axes[0].axvline(np.mean(all_widths), color='red', linestyle='--', label=f'Rata-rata: {np.mean(all_widths):.0f}px')
axes[0].legend()

# Histogram tinggi box
axes[1].hist(all_heights, bins=50, color='#3cb44b', edgecolor='white')
axes[1].set_title('Distribusi Tinggi Bounding Box')
axes[1].set_xlabel('Tinggi (pixel)')
axes[1].axvline(np.mean(all_heights), color='red', linestyle='--', label=f'Rata-rata: {np.mean(all_heights):.0f}px')
axes[1].legend()

# Jumlah objek per gambar
axes[2].hist(img_object_counts, bins=range(1, max(img_object_counts)+2),
             color='#f58231', edgecolor='white', align='left')
axes[2].set_title('Jumlah Objek per Gambar')
axes[2].set_xlabel('Jumlah Objek')
axes[2].set_ylabel('Frekuensi')
axes[2].axvline(np.mean(img_object_counts), color='red', linestyle='--',
                label=f'Rata-rata: {np.mean(img_object_counts):.1f}')
axes[2].legend()

plt.suptitle('EDA — Analisis Ukuran Bounding Box', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 45)
print("        RINGKASAN EDA PASCAL VOC")
print("=" * 45)
print(f"  Total gambar          : {len(anns):,}")
print(f"  Total objek           : {len(all_labels):,}")
print(f"  Jumlah class          : {len(label_counts)}")
print(f"  Rata-rata obj/gambar  : {np.mean(img_object_counts):.1f}")
print(f"  Max obj dalam 1 gambar: {max(img_object_counts)}")
print()
print("  Class terbanyak:")
for label, count in label_counts.most_common(5):
    bar = '█' * (count // 500)
    print(f"    {label:<15} {count:>6,}  {bar}")
print()
print("  Ukuran bounding box:")
print(f"    Lebar  — rata-rata {np.mean(all_widths):.0f}px, max {max(all_widths)}px")
print(f"    Tinggi — rata-rata {np.mean(all_heights):.0f}px, max {max(all_heights)}px")
print("=" * 45)

In [ ]:
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]
COLORS = [
    '#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231',
    '#911eb4', '#42d4f4', '#f032e6', '#bfef45', '#fabed4',
    '#469990', '#dcbeff', '#9A6324', '#fffac8', '#800000',
    '#aaffc3', '#808000', '#ffd8b1', '#000075', '#a9a9a9'
]

In [ ]:
def visualize_gt(img_path, xml_path, class_list=None):
    img     = Image.open(img_path).convert('RGB')
    objects = parse_xml(xml_path)

    if class_list is None:
        class_list = VOC_CLASSES
    color_map = {c: COLORS[i % len(COLORS)] for i, c in enumerate(class_list)}

    fig, ax = plt.subplots(1, figsize=(10, 8))
    ax.imshow(img)
    ax.axis('off')

    for (label, xmin, ymin, xmax, ymax) in objects:
        color = color_map.get(label, '#ffffff')
        w, h  = xmax - xmin, ymax - ymin
        rect  = patches.Rectangle(
            (xmin, ymin), w, h,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(xmin, ymin - 5, label,
               fontsize=10, color='white', fontweight='bold',
               bbox=dict(facecolor=color, alpha=0.8, pad=2, edgecolor='none'))

    plt.tight_layout()
    plt.show()

In [ ]:
sample_100 = random.sample(imgs, 1000)
batch = sample_100[:20]

for img_p in batch:
    name  = os.path.splitext(os.path.basename(img_p))[0]
    xml_p = os.path.join(ANN_DIR, name + '.xml')

    visualize_gt(img_p, xml_p)

print(f"Total sample: {len(sample_100)} | Ditampilkan: {len(batch)}")

In [ ]:
BASE = '/content/voc_yolo'
if os.path.exists(BASE):
    shutil.rmtree(BASE)
for folder in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(os.path.join(BASE, folder), exist_ok=True)

def convert_and_copy(img_list, split_name):
    ok, skip = 0, 0
    for img_p in img_list:
        name  = os.path.splitext(os.path.basename(img_p))[0]
        xml_p = os.path.join(ANN_DIR, name + '.xml')
        if not os.path.exists(xml_p):
            skip += 1
            continue
        size = root.find("size")
        w = int(size.find("width").text)
        h = int(size.find("height").text)
        objects   = parse_xml(xml_p)
        lines     = []
        for (label, xmin, ymin, xmax, ymax) in objects:
            if label not in VOC_CLASSES:
                continue
            class_id = VOC_CLASSES.index(label)
            cx = ((xmin + xmax) / 2) / w
            cy = ((ymin + ymax) / 2) / h
            bw = (xmax - xmin) / w
            bh = (ymax - ymin) / h
            lines.append(f"{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        txt_path = os.path.join(BASE, f'labels/{split_name}/{name}.txt')
        with open(txt_path, 'w') as f:
            f.write('\n'.join(lines))
        shutil.copy(img_p, os.path.join(BASE, f'images/{split_name}/{name}.jpg'))
        ok += 1
    print(f"{split_name}: {ok} gambar, {skip} dilewati")

random.seed(42)
all_imgs = glob.glob(os.path.join(IMG_DIR, '*.jpg'))
random.shuffle(all_imgs)
sample     = all_imgs[:1000]
split      = int(len(sample) * 0.8)
convert_and_copy(sample[:split], 'train')
convert_and_copy(sample[split:], 'val')

In [ ]:
yaml_content = """
path: /content/voc_yolo
train: images/train
val: images/val

nc: 20
names: ['aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
        'bus', 'car', 'cat', 'chair', 'cow',
        'diningtable', 'dog', 'horse', 'motorbike', 'person',
        'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor']
"""

yaml_path = '/content/voc100.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print("YAML siap!")

In [ ]:
model = YOLO('yolov8n.pt')

results = model.train(
    data=yaml_path,
    epochs=50,       # naikkan jadi 50
    imgsz=416,
    batch=16,        # naikkan jadi 16 karena data lebih banyak
    name='voc1000',
    verbose=True
)
# ← langsung simpan di sini, jangan pisah cell!
SAVE_DIR = '/content/drive/MyDrive/imageeeeee tes/voc1000_results'
if os.path.exists(SAVE_DIR):
    shutil.rmtree(SAVE_DIR)
shutil.copytree('/content/runs/detect/voc1000', SAVE_DIR)
print("✅ Model tersimpan ke Drive!")
print("Training selesai!")

In [ ]:
import os
print(os.listdir('/content/runs/detect'))

In [ ]:
# Lihat hasil training
results_dir = glob.glob('/content/runs/detect/voc1000')
print("Model tersimpan di:", results_dir)

In [ ]:
from IPython.display import Image, display
base = '/content/runs/detect/voc1000'

print("=== HASIL TRAINING ===")
display(Image(filename=f"{base}/results.png"))

print("\n=== CONFUSION MATRIX ===")
display(Image(filename=f"{base}/confusion_matrix_normalized.png"))

print("\n=== PREDIKSI VALIDASI ===")
display(Image(filename=f"{base}/val_batch0_pred.jpg"))

In [ ]:
model = YOLO('/content/runs/detect/voc1000/weights/best.pt')
metrics = model.val(data=yaml_path, verbose=True)

print("\n" + "="*45)
print("        HASIL EVALUASI MODEL")
print("="*45)
print(f"  mAP@0.5        : {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95   : {metrics.box.map:.4f}")
print(f"  Precision      : {metrics.box.mp:.4f}")
print(f"  Recall         : {metrics.box.mr:.4f}")
print("="*45)

In [ ]:
# Load model dari Drive
model = YOLO('/content/drive/MyDrive/imageeeeee tes/voc1000_results/weights/best.pt')
print("Model berhasil diload!")

In [ ]:
import gradio as gr
model_gradio = YOLO('/content/drive/MyDrive/imageeeeee tes/voc1000_results/weights/best.pt')

def detect_objects(image):
    results = model_gradio.predict(source=image, conf=0.25)
    return results[0].plot()

demo = gr.Interface(
    fn=detect_objects,
    inputs=gr.Image(type='numpy', label='Upload Gambar'),
    outputs=gr.Image(type='numpy', label='Hasil Deteksi'),
    title='🎯 Object Detection — YOLOv8 Pascal VOC',
    description='Upload gambar, model akan mendeteksi objek secara otomatis!'
)

demo.launch(share=True)